# OmniJev: local classification

In VS Code, use the project **.venv (Python 3.13.0)** kernel and click **Run All**.
The model is already downloaded. The three cells below load it, classify examples, and let you try your own text.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display

repo_name = "Decision-Aware-Pruning-for-Efficient-Typed-Decision-Models"
candidates = [Path.cwd(), Path.cwd() / repo_name, Path.cwd() / "Final_Project" / repo_name]
PROJECT = next((p for p in candidates if (p / "omnijev_demo.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Open the project folder in VS Code, then run this notebook.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import omnijev_demo
importlib.reload(omnijev_demo)
from omnijev_demo import load_model, classify_text, summarize_answers

if "model" not in globals():
    model = load_model(PROJECT)
else:
    print("Reusing the loaded OmniJev model.")

W1103 18:05:29.051000 1054 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loaded real OmniJev-0.8B on cpu using torch.float32 in 7.4s.


## Classify a few messages

`LABELS` gives the categories and their meanings. Edit or add messages in `TEXTS`.
The table shows each category's probability, including the model's option to abstain.

In [2]:
LABELS = {
    "billing": "Payments, charges, invoices, refunds.",
    "technical_support": "Broken app, bugs, software problems.",
    "account_access": "Passwords, sign in, access problems.",
}
TEXTS = [
    "I was charged twice for my subscription this month. Can you help me get a refund?",
    "The app crashes every time I try to open a saved document.",
    "I forgot my password and cannot sign in to my account.",
]

runs = []
rows = []
for text in TEXTS:
    result = classify_text(model, text, LABELS)
    runs.append(result)
    answer = result["answers"]["category"]
    probabilities = {**answer["probabilities"], "[abstain]": answer["abstain"]}
    rows.append({"message": text, "prediction": max(probabilities, key=probabilities.get), **probabilities})

display(pd.DataFrame(rows).round(4))

/Users/tiendeelin/Desktop/Penn_Courses/ESE_6800_Systems_for_Machine_Learning/Final_Project/Decision-Aware-Pruning-for-Efficient-Typed-Decision-Models/.venv/lib/python3.13/site-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


,message,prediction,billing,technical_support,account_access,[abstain]
0,I was charged twice for my subscription this m...,billing,0.9933,0.0002,0.0033,0.0032
1,The app crashes every time I try to open a sav...,technical_support,0.0005,0.8195,0.1598,0.0202
2,I forgot my password and cannot sign in to my ...,account_access,0.0001,0.0026,0.9919,0.0054


## Try your own text

Change `TEXT`, then click **▶** beside this cell. It reuses the model already loaded above.

In [4]:
TEXT = "I am a stupid big boy and I am hungry"

result = classify_text(model, TEXT, LABELS)
display(summarize_answers(result)[["option", "probability"]].sort_values("probability", ascending=False))

,option,probability
3,[abstain],0.9566
2,account_access,0.0325
1,technical_support,0.0081
0,billing,0.0028
